# Vigor da vegetação por cultura com Sentinel-2 e Google Earth Engine
Série mensal 2019–2026 de NDVI / NDRE / NDMI em talhões de 8 usos do solo (cacau, café, banana, coco, milho irrigado, eucalipto, pasto e IPF), cruzada com chuva (CHIRPS).

**Perguntas:** (1) como cada cultura se comporta ao longo do ano? (2) quando a vegetação saiu do normal? (3) qual cultura responde mais à chuva?

> Rode primeiro `src/anonymize.py` (gera `data/talhoes_anonimo.geojson`). A análise usa a geometria real (privada); só os resultados anonimizados são publicados.

In [ ]:
# %pip install -q earthengine-api geemap pandas numpy matplotlib   # descomente no Colab
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))
import pandas as pd
import config as C, anonymize, gee_extract as G, analysis as A
C.OUT.mkdir(exist_ok=True); (C.OUT / "privado").mkdir(exist_ok=True)

## 1. Dados e anonimização

In [ ]:
anonymize.anonimizar()                 # gera data/talhoes_anonimo.geojson
areas = A.carregar_areas()
areas.groupby("cultura").area_ha.agg(["count", "sum"]).round(0).sort_values("sum", ascending=False)

In [ ]:
# Área nominal (escrita no nome do polígono) vs. área mapeada
A.area_nominal_vs_mapeada(areas)

## 2. Extração no Earth Engine
Sentinel-2 SR Harmonizado + máscara de nuvem **Cloud Score+** (`cs_cdf ≥ 0.60`), composto mensal por mediana, média por talhão (10 m).

In [ ]:
G.iniciar()                            # defina EE_PROJECT em src/config.py
fc = G.talhoes_fc()
cache = C.OUT / "privado" / "s2_mensal.csv"
if cache.exists():
    df = pd.read_csv(cache, parse_dates=["data"])
else:
    df = G.serie_s2(fc)
    df.to_csv(cache, index=False)
df.head()
df = df[df.data <= pd.to_datetime(C.FIM)]   # sem mês incompleto


In [ ]:
cache_c = C.OUT / "privado" / "clima_mensal.csv"
if cache_c.exists():
    clima = pd.read_csv(cache_c, parse_dates=["data"])
else:
    clima = G.clima_mensal(fc); clima.to_csv(cache_c, index=False)
clima.tail()

In [ ]:
# Dados publicáveis (sem coordenadas, só IDs anônimos)
df.drop(columns=["n_cenas"], errors="ignore").to_csv(C.OUT / "ndvi_ndre_ndmi_mensal.csv", index=False)
clima.to_csv(C.OUT / "clima_mensal.csv", index=False)
print("cenas válidas por mês (mediana):", df.groupby("data").n_cenas.first().median())

## 3. Perfil sazonal por cultura

In [ ]:
# Controle de qualidade: nuvem/névoa residual
df, removidos = A.filtrar_espurios(df)            # quedas de 1 mês cercadas por meses normais
print(len(removidos), "pontos talhão-mês removidos (", round(100*len(removidos)/len(df), 1), "%)")
sc = A.serie_cultura(df, areas)
sc, meses_sinc = A.filtrar_eventos_sincronos(sc)  # várias culturas caindo juntas = atmosfera
print("meses descartados:", [d.strftime("%Y-%m") for d in meses_sinc])
clim = A.climatologia(sc)
display(A.amplitude_sazonal(clim))
A.fig_curvas(sc, destino=C.OUT / "01_curvas_ndvi.png");


## 4. Anomalias: quando a vegetação fugiu do normal
z-score do NDVI mensal contra a climatologia 2019–2025.

In [ ]:
# Eucalipto e IPF passam por ciclo de corte/rebrota: ficam fora das anomalias climáticas
sa = sc[~sc.cultura.isin(A.CULTURAS_MANEJO)]
ac = A.anomalia_cultura(sa)
A.fig_anomalias(ac, destino=C.OUT / "02_anomalias.png");


## 5. Sensibilidade à chuva
Correlação entre o z-score do NDVI e o z-score da chuva, testando defasagens de 0 a 3 meses.

In [ ]:
sens = A.sensibilidade_hidrica(ac, clima)
display(sens)
A.fig_sensibilidade(sens, destino=C.OUT / "03_sensibilidade_chuva.png");

## 5b. O manejo do café no NDVI
Calendário do conilon no ES (Incaper/Agrolink): colheita abr–jun, poda mai–ago (pico em jul), fim dos tratos em set, brotação/florada depois.

In [ ]:
A.fig_cafe_manejo(df, destino=C.OUT / "05_cafe_manejo.png");

## 6. Mapa esquemático (sem base cartográfica)

In [ ]:
ult = df[df.data == df.data.max()].set_index("id").NDVI.to_dict()
A.fig_mapa_esquematico(C.GEOJSON_ANON, ult, f"NDVI em {df.data.max():%m/%Y}",
                       destino=C.OUT / "04_mapa_ndvi.png");

## 7. Mapa interativo com base de satélite (**só se tiver autorização**)

In [ ]:
PUBLICAR_MAPA_REAL = False     # True só com autorização do proprietário
import geemap
m = geemap.Map(); m.add_basemap("SATELLITE")
comp = G.composto_mensal(fc.geometry().bounds(), 2026, 8).select("NDVI")
m.addLayer(comp.clip(fc), {"min": .2, "max": .9, "palette": ["#d73027", "#fee08b", "#1a9850"]}, "NDVI ago/2026")
m.addLayer(fc.style(color="white", fillColor="00000000", width=1), {}, "Talhões")
m.centerObject(fc, 12)
if PUBLICAR_MAPA_REAL:
    m.to_html(C.OUT / "mapa_interativo.html")
m

## 8. Conclusões
- **Sazonalidade:** pasto (0,43→0,67, pico em março) e café (pico em maio) variam mais; banana (0,78–0,86) e coco (0,64–0,74) são quase planos.
- **Chuva:** só pasto (r = +0,41) e café (r = +0,35) respondem com clareza; no café o motivo provável é o manejo (colheita e poda), não a chuva. Cacau, banana e milho ficam em torno de +0,2 e o coco (irrigado) não responde.
- **Manejo visível do espaço:** eucalipto e IPF caíram a NDVI ≈ 0,18 em meados de 2024 e voltaram a ≈ 0,8 em 2025 (corte e rebrota).
- **Cadastro:** áreas nominais diferem da mapeada em até +46 % (Cacau-01).
- **Limitações:** média por polígono inclui estradas e bordas; NDVI satura em dossel denso; sem validação de campo; n ≈ 85 meses por cultura, então correlações de ~0,2 são fracas.
